# MURA DINOv2 Attention-TransMIL end-to-end + staged backbone unfreeze

This notebook trains a study-level MIL model directly from MURA images. The main model is `EndToEndMURA`: DINOv2 extracts image features inside `forward`, then `SimpleTransMIL` pools all images in one study and emits one study-level logit.

Differences from `CP_7_05`:
- no precomputed frozen embeddings;
- DINOv2 is part of the trainable model wrapper;
- staged backbone unfreezing and train augmentations follow `CP_7_02_MURA_DINOv2_MLflow_Colab_with_enhanced.ipynb`;
- validation logs confusion matrix pieces and prediction positive rate so kappa failures are visible immediately.


In [12]:
# Install only missing Colab dependencies.
import importlib.util
import subprocess
import sys

missing = [pkg for pkg in ["mlflow", "boto3", "psycopg2"] if importlib.util.find_spec(pkg) is None]
if missing:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "mlflow>=2.15.1,<3",
        "boto3>=1.34",
        "psycopg2-binary>=2.9",
    ])


In [13]:
from __future__ import annotations

import json
import math
import os
import random
import shutil
import sys
import time
import zipfile
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import mlflow
import mlflow.pytorch
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm
from mlflow.tracking import MlflowClient

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram_gb:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
device: cuda
gpu: NVIDIA RTX PRO 6000 Blackwell Server Edition
vram_gb: 95.0


In [14]:
# Colab/runtime constants.
VPS_HOST = ""
MLFLOW_TRACKING_URI = f"http://{VPS_HOST}:8094"
EXPERIMENT_NAME = "project2025-mura-dinov2-attention-trans-mil-unfreeze-bacbone"
REGISTERED_MODEL_NAME = "mura_dinov2_attention_trans-mil-unfreeze-backbone"
RUN_NAME = "dinov2_attention_trans_mil_unfreeze_backbone"
MODEL_VERSION_ALIAS = "prd"
MLFLOW_ENABLED = True
REGISTER_MODEL = True

SEED = 42
BACKBONE_NAME = "dinov2_vitl14"
IMAGE_SIZE = 448
NUM_WORKERS = 2

EPOCHS = 30
ACCUMULATION_STEPS = 8
HEAD_LR = 3e-4
BACKBONE_LR = 8e-6
WEIGHT_DECAY = 1e-2
SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 1
SCHEDULER_MIN_LR = 1e-7
DROPOUT = 0.1
MAX_GRAD_NORM = 1.0
THRESHOLD = 0.5
INTERNAL_VAL_SIZE = 0.10

# Same staged unfreeze schedule as CP_7_02.
UNFREEZE_SCHEDULE = {
    6: 4,
    9: 8,
    12: 12,
}

USE_AMP = True
RESUME = True
RUN_TRAINING = True
COPY_ZIP_TO_LOCAL_DISK = True
ONLY_ANATOMY = None
LIMIT_STUDIES = None

if IN_COLAB:
    DRIVE_ROOT = Path("/content/drive/MyDrive/Project2025/attention_mil_dinov2-transmil-unfreeze")
    MURA_ZIP_PATH = Path("/content/drive/MyDrive/Project2025/MURA-v1.1-resized-448x448.zip")
    LOCAL_ZIP_PATH = Path("/content/MURA-v1.1-resized-448x448.zip")
    DATA_EXTRACT_DIR = Path("/content/mura_attention_transmil_unfreeze_data")
else:
    DRIVE_ROOT = Path("../data/mura_attention_transmil_unfreeze")
    MURA_ZIP_PATH = Path("../MURA-v1.1-resized-448x448.zip")
    LOCAL_ZIP_PATH = MURA_ZIP_PATH
    DATA_EXTRACT_DIR = Path("../")

DATA_ROOT = DATA_EXTRACT_DIR / "MURA-v1.1-resized-448x448"
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints"
RESULTS_DIR = DRIVE_ROOT / "results"
ARTIFACT_DIR = DRIVE_ROOT / "artifacts"

for path in [DRIVE_ROOT, CHECKPOINT_DIR, RESULTS_DIR, ARTIFACT_DIR, DATA_EXTRACT_DIR]:
    path.mkdir(parents=True, exist_ok=True)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


seed_everything(SEED)
if MLFLOW_ENABLED:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    print("MLflow tracking URI:", mlflow.get_tracking_uri())


MLflow tracking URI: http://:8094


## Data setup

Source `train` is split into `train/internal_val` by study inside anatomy and label buckets. Official MURA `valid` is held out as `test`.


In [15]:
def resolve_zip_path() -> Path:
    candidates = [MURA_ZIP_PATH, LOCAL_ZIP_PATH, Path("/content/MURA-v1.1-resized-448x448.zip")]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"MURA zip not found. Expected: {MURA_ZIP_PATH}")


def prepare_zip_for_extraction(zip_path: Path) -> Path:
    if not IN_COLAB or not COPY_ZIP_TO_LOCAL_DISK:
        return zip_path
    if zip_path.resolve() == LOCAL_ZIP_PATH.resolve():
        return LOCAL_ZIP_PATH
    if LOCAL_ZIP_PATH.exists() and LOCAL_ZIP_PATH.stat().st_size == zip_path.stat().st_size:
        print("using existing local zip:", LOCAL_ZIP_PATH)
        return LOCAL_ZIP_PATH
    print(f"copying zip to local disk: {zip_path} -> {LOCAL_ZIP_PATH}")
    start = time.time()
    LOCAL_ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(zip_path, LOCAL_ZIP_PATH)
    print(f"copied in {time.time() - start:.1f}s")
    return LOCAL_ZIP_PATH


def find_mura_root(extract_dir: Path) -> Path:
    candidates = [DATA_ROOT, extract_dir / "MURA-v1.1-resized-448x448", extract_dir / "MURA-v1.1", extract_dir]
    candidates.extend([p for p in extract_dir.rglob("*") if p.is_dir() and p.name.startswith("MURA-v1.1")])
    for candidate in candidates:
        if (candidate / "train").is_dir() and (candidate / "valid").is_dir():
            return candidate
    raise FileNotFoundError(f"Could not locate MURA train/valid under {extract_dir}")


if not DATA_ROOT.exists():
    zip_path = prepare_zip_for_extraction(resolve_zip_path())
    print("extracting", zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(DATA_EXTRACT_DIR)

DATA_ROOT = find_mura_root(DATA_EXTRACT_DIR)
print("ready:", DATA_ROOT)


ready: /content/mura_attention_transmil_unfreeze_data/MURA-v1.1-resized-448x448


In [16]:
def parse_study_path(study_path: Path) -> dict:
    parts = study_path.parts
    split = parts[-4]
    anatomy = parts[-3]
    patient_id = parts[-2]
    study_id = parts[-1]
    label = 1 if "positive" in study_id else 0
    study_key = f"{anatomy}/{patient_id}/{study_id}"
    return {
        "split": split,
        "mil_split": split,
        "anatomy": anatomy,
        "patient_id": patient_id,
        "study_id": study_id,
        "study_key": study_key,
        "label": label,
        "path": str(study_path),
    }


def list_images(study_path: Path) -> list[Path]:
    suffixes = {".png", ".jpg", ".jpeg"}
    return sorted(path for path in study_path.iterdir() if path.suffix.lower() in suffixes)


def build_study_dataframe(root_dir: Path) -> pd.DataFrame:
    records = []
    for split in ["train", "valid"]:
        split_dir = root_dir / split
        if not split_dir.exists():
            continue
        for anatomy_dir in sorted(split_dir.iterdir()):
            if not anatomy_dir.is_dir():
                continue
            for patient_dir in sorted(anatomy_dir.iterdir()):
                if not patient_dir.is_dir():
                    continue
                for study_dir in sorted(patient_dir.iterdir()):
                    if study_dir.is_dir():
                        records.append(parse_study_path(study_dir))
    frame = pd.DataFrame(records)
    if ONLY_ANATOMY is not None:
        frame = frame[frame["anatomy"] == ONLY_ANATOMY]
    frame = frame.sort_values(["split", "anatomy", "patient_id", "study_id"]).reset_index(drop=True)
    if LIMIT_STUDIES is not None:
        frame = frame.head(LIMIT_STUDIES)
    return frame


def assign_mil_splits(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    frame["mil_split"] = np.where(frame["split"] == "valid", "test", frame["split"])
    train_source = frame[frame["split"] == "train"].copy()
    train_indices = []
    internal_val_indices = []

    for (_, _), part in train_source.groupby(["anatomy", "label"], sort=True):
        if len(part) < 2:
            train_indices.extend(part.index.tolist())
            continue
        splitter = GroupShuffleSplit(n_splits=1, test_size=INTERNAL_VAL_SIZE, random_state=SEED)
        local_train, local_val = next(splitter.split(part, groups=part["study_key"]))
        train_indices.extend(part.iloc[local_train].index.tolist())
        internal_val_indices.extend(part.iloc[local_val].index.tolist())

    frame.loc[train_indices, "mil_split"] = "train"
    frame.loc[internal_val_indices, "mil_split"] = "internal_val"
    return frame.sort_values(["mil_split", "anatomy", "patient_id", "study_id"]).reset_index(drop=True)


studies_df = assign_mil_splits(build_study_dataframe(DATA_ROOT))
for split_name, part in studies_df.groupby("mil_split"):
    part.to_csv(ARTIFACT_DIR / f"split_{split_name}.csv", index=False)

print(studies_df.shape)
display(studies_df.groupby(["mil_split", "anatomy", "label"]).size().rename("n").reset_index())
print("original folder split counts:")
display(studies_df.groupby(["split", "mil_split"]).size().rename("n").reset_index())


(14656, 8)


,mil_split,anatomy,label,n
0,internal_val,XR_ELBOW,0,110
1,internal_val,XR_ELBOW,1,66
2,internal_val,XR_FINGER,0,128
3,internal_val,XR_FINGER,1,66
4,internal_val,XR_FOREARM,0,59
5,internal_val,XR_FOREARM,1,29
6,internal_val,XR_HAND,0,150
7,internal_val,XR_HAND,1,53
8,internal_val,XR_HUMERUS,0,33
9,internal_val,XR_HUMERUS,1,28


original folder split counts:


,split,mil_split,n
0,train,internal_val,1352
1,train,train,12105
2,valid,test,1199


## Augmentations and study loaders

Train augmentations match `CP_7_02`: horizontal flip, rotation, affine translate/scale, and color jitter after square pad resize.


In [17]:
class SquarePadResize:
    def __init__(self, size: int, fill: int = 0):
        self.size = size
        self.fill = fill

    def __call__(self, image: Image.Image) -> Image.Image:
        image = image.convert("RGB")
        width, height = image.size
        scale = self.size / max(width, height)
        new_width = max(1, int(round(width * scale)))
        new_height = max(1, int(round(height * scale)))
        image = image.resize((new_width, new_height), Image.BICUBIC)
        canvas = Image.new("RGB", (self.size, self.size), color=(self.fill, self.fill, self.fill))
        canvas.paste(image, ((self.size - new_width) // 2, (self.size - new_height) // 2))
        return canvas


MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    SquarePadResize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10, fill=0),
    transforms.RandomAffine(degrees=0, translate=(0.04, 0.04), scale=(0.94, 1.06), fill=0),
    transforms.ColorJitter(brightness=0.10, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

eval_transform = transforms.Compose([
    SquarePadResize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])


class StudyImageDataset(Dataset):
    def __init__(self, manifest: pd.DataFrame, split: str, transform, anatomy: str | None = None):
        frame = manifest[manifest["mil_split"] == split].copy()
        if anatomy is not None:
            frame = frame[frame["anatomy"] == anatomy]
        if frame.empty:
            raise ValueError(f"No rows for split={split}, anatomy={anatomy}")

        rows = []
        for _, row in frame.iterrows():
            image_paths = list_images(Path(row["path"]))
            if image_paths:
                item = row.to_dict()
                item["image_paths"] = [str(path) for path in image_paths]
                item["num_images"] = len(image_paths)
                rows.append(item)
        if not rows:
            raise ValueError(f"No image files for split={split}, anatomy={anatomy}")

        self.frame = pd.DataFrame(rows).reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx: int):
        row = self.frame.iloc[idx]
        tensors = []
        for path_str in row["image_paths"]:
            with Image.open(path_str) as image:
                tensors.append(self.transform(image))
        images = torch.stack(tensors, dim=0)
        label = torch.tensor([float(row["label"])], dtype=torch.float32)
        meta = {
            "study_key": row["study_key"],
            "anatomy": row["anatomy"],
            "patient_id": row["patient_id"],
            "study_id": row["study_id"],
            "num_images": int(row["num_images"]),
        }
        return images, label, meta


def collate_one(batch):
    if len(batch) != 1:
        raise ValueError("End-to-end MIL uses batch_size=1 because bags have variable N")
    return batch[0]


def seed_worker(worker_id: int) -> None:
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)


loader_generator = torch.Generator().manual_seed(SEED)
train_ds = StudyImageDataset(studies_df, split="train", transform=train_transform, anatomy=ONLY_ANATOMY)
internal_val_ds = StudyImageDataset(studies_df, split="internal_val", transform=eval_transform, anatomy=ONLY_ANATOMY)
test_ds = StudyImageDataset(studies_df, split="test", transform=eval_transform, anatomy=ONLY_ANATOMY)

train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_one,
    worker_init_fn=seed_worker,
    generator=loader_generator,
)
internal_val_loader = DataLoader(
    internal_val_ds,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_one,
)
test_loader = DataLoader(
    test_ds,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_one,
)

xb, yb, mb = next(iter(train_loader))
print({
    "study_tensor": tuple(xb.shape),
    "label": yb.tolist(),
    "meta": mb,
    "train_studies": len(train_ds),
    "internal_val_studies": len(internal_val_ds),
    "test_studies": len(test_ds),
})


{'study_tensor': (2, 3, 448, 448), 'label': [0.0], 'meta': {'study_key': 'XR_FOREARM/patient09684/study1_negative', 'anatomy': 'XR_FOREARM', 'patient_id': 'patient09684', 'study_id': 'study1_negative', 'num_images': 2}, 'train_studies': 12105, 'internal_val_studies': 1352, 'test_studies': 1199}


## Model wrapper

`EndToEndMURA` is the primary model. It receives all images from one study, extracts one DINOv2 feature per image, and feeds the resulting bag to `SimpleTransMIL`.


In [18]:
def expected_embed_dim_from_backbone_name(backbone_name: str) -> int:
    known_dims = {
        "dinov2_vits14": 384,
        "dinov2_vitb14": 768,
        "dinov2_vitl14": 1024,
        "dinov2_vitg14": 1536,
    }
    return known_dims.get(backbone_name, 1024)


def feature_tensor_from_backbone_output(output) -> torch.Tensor:
    if isinstance(output, dict):
        if "x_norm_clstoken" in output:
            return output["x_norm_clstoken"]
        if "last_hidden_state" in output:
            return output["last_hidden_state"][:, 0, :]
        raise ValueError(f"Cannot infer DINO feature tensor from keys: {list(output)}")
    if hasattr(output, "last_hidden_state"):
        return output.last_hidden_state[:, 0, :]
    if isinstance(output, (tuple, list)):
        return output[0]
    return output


class SimpleTransMIL(nn.Module):
    def __init__(self, embed_dim=1024, hidden_dim=512, num_layers=2, num_heads=8, dropout=0.1):
        super().__init__()
        self.input_norm = nn.LayerNorm(embed_dim)
        self.dim_reduction = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.cls_token = nn.Parameter(torch.zeros(1, 1, hidden_dim))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_norm = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        if features.ndim == 3:
            features = features.squeeze(0)
        if features.ndim != 2:
            raise ValueError(f"Expected features [N, D], got {tuple(features.shape)}")
        h = self.dim_reduction(self.input_norm(features)).unsqueeze(0)
        cls_tokens = self.cls_token.expand(h.size(0), -1, -1)
        h = torch.cat((cls_tokens, h), dim=1)
        h = self.transformer(h)
        study_embedding = self.output_norm(h[:, 0, :])
        return self.classifier(study_embedding).view(-1)


class EndToEndMURA(nn.Module):
    def __init__(self, dinov2_model, transmil_model):
        super().__init__()
        self.backbone = dinov2_model
        self.mil_head = transmil_model

    def forward(self, x):
        # x: [N, 3, 448, 448]
        features = self.backbone(x)  # -> [N, 1024] for torch.hub DINOv2
        features = feature_tensor_from_backbone_output(features)
        logits = self.mil_head(features)  # -> [1]
        return logits.view(-1)

    def freeze_backbone(self) -> None:
        for parameter in self.backbone.parameters():
            parameter.requires_grad = False

    def transformer_blocks(self) -> list[nn.Module]:
        if hasattr(self.backbone, "blocks"):
            return list(self.backbone.blocks)
        if hasattr(self.backbone, "encoder") and hasattr(self.backbone.encoder, "layer"):
            return list(self.backbone.encoder.layer)
        if hasattr(self.backbone, "model") and hasattr(self.backbone.model, "blocks"):
            return list(self.backbone.model.blocks)
        raise AttributeError("Unsupported DINO backbone: cannot find transformer blocks")

    def set_trainable_last_n_blocks(self, n_blocks: int) -> None:
        self.freeze_backbone()
        if n_blocks <= 0:
            return
        blocks = self.transformer_blocks()
        for block in blocks[-n_blocks:]:
            for parameter in block.parameters():
                parameter.requires_grad = True


def trainable_param_count(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def total_param_count(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def n_unfrozen_blocks_for_epoch(epoch: int) -> int:
    n_blocks = 0
    for start_epoch, blocks in sorted(UNFREEZE_SCHEDULE.items()):
        if epoch >= start_epoch:
            n_blocks = int(blocks)
    return n_blocks


def configure_trainable(model: EndToEndMURA, epoch: int) -> int:
    n_blocks = n_unfrozen_blocks_for_epoch(epoch)
    model.set_trainable_last_n_blocks(n_blocks)
    for parameter in model.mil_head.parameters():
        parameter.requires_grad = True
    return n_blocks


def make_optimizer(model: EndToEndMURA) -> torch.optim.Optimizer:
    backbone_params = []
    head_params = []
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if name.startswith("backbone."):
            backbone_params.append(parameter)
        else:
            head_params.append(parameter)
    groups = []
    if backbone_params:
        groups.append({"params": backbone_params, "lr": BACKBONE_LR, "name": "backbone"})
    if head_params:
        groups.append({"params": head_params, "lr": HEAD_LR, "name": "mil_head"})
    return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)


def lr_summary(optimizer) -> str:
    return ", ".join(f"{group.get('name', i)}={group['lr']:.2e}" for i, group in enumerate(optimizer.param_groups))


In [19]:
print("loading", BACKBONE_NAME)
backbone = torch.hub.load("facebookresearch/dinov2", BACKBONE_NAME)
embed_dim = int(getattr(backbone, "embed_dim", 0)) or expected_embed_dim_from_backbone_name(BACKBONE_NAME)
mil_head = SimpleTransMIL(embed_dim=embed_dim, dropout=DROPOUT)
model = EndToEndMURA(backbone, mil_head).to(DEVICE)
configure_trainable(model, epoch=1)

train_frame = train_ds.frame
positives = int((train_frame["label"] == 1).sum())
negatives = int((train_frame["label"] == 0).sum())
pos_weight_value = negatives / max(positives, 1)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_value], dtype=torch.float32, device=DEVICE))

print({
    "embed_dim": embed_dim,
    "total_params_m": round(total_param_count(model) / 1e6, 2),
    "trainable_params_m_epoch_1": round(trainable_param_count(model) / 1e6, 2),
    "pos_weight_bce": pos_weight_value,
})


loading dinov2_vitl14


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


{'embed_dim': 1024, 'total_params_m': 311.2, 'trainable_params_m_epoch_1': 6.83, 'pos_weight_bce': 1.5998711340206186}


/tmp/ipykernel_650/3180166094.py:45: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


## Metrics and training utilities


In [20]:
def binary_metrics(labels, logits, threshold=0.5):
    y_true = np.asarray(labels, dtype=np.int64).reshape(-1)
    logits_arr = np.asarray(logits, dtype=np.float32).reshape(-1)
    probs = 1.0 / (1.0 + np.exp(-logits_arr))
    y_pred = (probs >= threshold).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    out = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "kappa": float(cohen_kappa_score(y_true, y_pred)),
        "specificity": float(tn / max(tn + fp, 1)),
        "roc_auc": float("nan"),
        "pr_auc": float("nan"),
        "label_positive_rate": float(np.mean(y_true == 1)),
        "pred_positive_rate": float(np.mean(y_pred == 1)),
        "tn": float(tn),
        "fp": float(fp),
        "fn": float(fn),
        "tp": float(tp),
    }
    if len(np.unique(y_true)) > 1:
        out["roc_auc"] = float(roc_auc_score(y_true, probs))
    if np.any(y_true == 1):
        out["pr_auc"] = float(average_precision_score(y_true, probs))
    if np.isfinite(out["roc_auc"]) and np.isfinite(out["pr_auc"]):
        out["composite"] = 0.8 * out["pr_auc"] + 0.2 * out["roc_auc"]
    else:
        out["composite"] = float("nan")
    return out


def best_threshold(labels, logits, metric="kappa"):
    best_t = 0.5
    best_m = binary_metrics(labels, logits, threshold=best_t)
    for threshold in np.linspace(0.05, 0.95, 91):
        metrics = binary_metrics(labels, logits, threshold=float(threshold))
        if metrics[metric] > best_m[metric]:
            best_t = float(threshold)
            best_m = metrics
    return best_t, best_m


def safe_float_metrics(metrics: dict) -> dict:
    out = {}
    for key, value in metrics.items():
        try:
            value = float(value)
        except (TypeError, ValueError):
            continue
        if np.isfinite(value):
            out[key] = value
    return out


def set_runtime_train_mode(model: EndToEndMURA, train: bool) -> None:
    model.train(train)
    if train and not any(parameter.requires_grad for parameter in model.backbone.parameters()):
        model.backbone.eval()


def run_epoch(loader, train: bool, optimizer=None, scaler=None):
    set_runtime_train_mode(model, train)
    total_loss = 0.0
    labels = []
    logits_all = []
    rows = []

    if train:
        optimizer.zero_grad(set_to_none=True)

    iterator = tqdm(loader, desc="train" if train else "eval", leave=False)
    for step, (images, label, meta) in enumerate(iterator, start=1):
        images = images.to(DEVICE, non_blocking=True)
        label = label.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train):
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP and DEVICE.type == "cuda"):
                logits = model(images)
                loss = criterion(logits.view(-1), label.view(-1))
                step_loss = loss / ACCUMULATION_STEPS if train else loss

            if train:
                scaler.scale(step_loss).backward()
                if step % ACCUMULATION_STEPS == 0 or step == len(loader):
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], MAX_GRAD_NORM)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)

        total_loss += float(loss.detach().cpu())
        label_value = float(label.item())
        logit_value = float(logits.detach().float().cpu().item())
        labels.append(label_value)
        logits_all.append(logit_value)
        rows.append({
            "study_key": meta["study_key"],
            "anatomy": meta["anatomy"],
            "label": int(label_value),
            "logit": logit_value,
            "prob": float(1.0 / (1.0 + math.exp(-logit_value))),
            "num_images": int(meta["num_images"]),
        })
        iterator.set_postfix(loss=total_loss / max(step, 1))

    return total_loss / max(len(loader), 1), labels, logits_all, pd.DataFrame(rows)


In [21]:
def training_config() -> dict:
    return {
        "backbone_name": BACKBONE_NAME,
        "image_size": IMAGE_SIZE,
        "embed_dim": embed_dim,
        "model_family": "dinov2_end_to_end_simple_transmil",
        "input_type": "raw study images [N,3,448,448]",
        "mil_pooling": "simple_transformer_cls_token",
        "epochs": EPOCHS,
        "accumulation_steps": ACCUMULATION_STEPS,
        "head_lr": HEAD_LR,
        "backbone_lr": BACKBONE_LR,
        "weight_decay": WEIGHT_DECAY,
        "dropout": DROPOUT,
        "max_grad_norm": MAX_GRAD_NORM,
        "threshold": THRESHOLD,
        "internal_val_size": INTERNAL_VAL_SIZE,
        "unfreeze_schedule": UNFREEZE_SCHEDULE,
        "scheduler": "ReduceLROnPlateau",
        "scheduler_monitor": "internal_val_composite",
        "checkpoint_monitor": "best_internal_val_kappa",
        "scheduler_factor": SCHEDULER_FACTOR,
        "scheduler_patience": SCHEDULER_PATIENCE,
        "scheduler_min_lr": SCHEDULER_MIN_LR,
        "pos_weight_bce": float(pos_weight_value),
        "only_anatomy": ONLY_ANATOMY,
        "train_studies": len(train_ds),
        "internal_val_studies": len(internal_val_ds),
        "test_studies_official_valid": len(test_ds),
        "augmentation": "hflip, rotation, affine, colorjitter",
        "split_policy": "source train -> train/internal_val 90/10 by study; source valid -> test",
    }


def save_checkpoint(path: Path, optimizer, scheduler, scaler, epoch: int, best_metric: float, history: list[dict], n_blocks: int, best_threshold_value: float) -> None:
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "best_metric": best_metric,
        "history": history,
        "n_unfrozen_blocks": n_blocks,
        "best_internal_val_threshold": best_threshold_value,
        "config": training_config(),
    }, path)


def load_checkpoint(path: Path, optimizer=None, scheduler=None, scaler=None) -> dict:
    checkpoint = torch.load(path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    if optimizer is not None and checkpoint.get("optimizer_state_dict") is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    if scheduler is not None and checkpoint.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    if scaler is not None and checkpoint.get("scaler_state_dict") is not None:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
    return checkpoint


def get_or_create_mlflow_experiment() -> str | None:
    if not MLFLOW_ENABLED:
        return None
    client = MlflowClient()
    existing = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    if existing is None:
        experiment_id = client.create_experiment(EXPERIMENT_NAME, artifact_location="mlflow-artifacts:/")
    else:
        experiment_id = existing.experiment_id
    mlflow.set_experiment(EXPERIMENT_NAME)
    return experiment_id


## Train and final official-valid evaluation


In [22]:
suffix = f"_{ONLY_ANATOMY}" if ONLY_ANATOMY else ""
last_checkpoint_path = CHECKPOINT_DIR / f"end_to_end_transmil_{BACKBONE_NAME}_{IMAGE_SIZE}{suffix}_last.pt"
best_checkpoint_path = CHECKPOINT_DIR / f"end_to_end_transmil_{BACKBONE_NAME}_{IMAGE_SIZE}{suffix}_best.pt"
history_path = RESULTS_DIR / f"end_to_end_transmil_{BACKBONE_NAME}_{IMAGE_SIZE}{suffix}_history.csv"
final_eval_path = RESULTS_DIR / f"end_to_end_transmil_{BACKBONE_NAME}_{IMAGE_SIZE}{suffix}_final_eval.csv"
test_predictions_path = RESULTS_DIR / f"end_to_end_transmil_{BACKBONE_NAME}_{IMAGE_SIZE}{suffix}_test_predictions.csv"
config_path = ARTIFACT_DIR / f"end_to_end_transmil_{BACKBONE_NAME}_{IMAGE_SIZE}{suffix}_config.json"
config_path.write_text(json.dumps(training_config(), indent=2, ensure_ascii=False))

EXPERIMENT_ID = get_or_create_mlflow_experiment()
history = []
best_metric = -float("inf")
best_threshold_from_internal_val = THRESHOLD
start_epoch = 1
optimizer = None
scheduler = None
current_blocks = None
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and DEVICE.type == "cuda")

if RESUME and last_checkpoint_path.exists():
    temp = torch.load(last_checkpoint_path, map_location="cpu")
    temp_epoch = int(temp.get("epoch", 0))
    start_epoch = temp_epoch + 1
    current_blocks = configure_trainable(model, max(temp_epoch, 1))
    optimizer = make_optimizer(model)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=SCHEDULER_FACTOR, patience=SCHEDULER_PATIENCE, min_lr=SCHEDULER_MIN_LR
    )
    checkpoint = load_checkpoint(last_checkpoint_path, optimizer=optimizer, scheduler=scheduler, scaler=scaler)
    history = list(checkpoint.get("history", []))
    best_metric = float(checkpoint.get("best_metric", -float("inf")))
    best_threshold_from_internal_val = float(checkpoint.get("best_internal_val_threshold", THRESHOLD))
    print(f"resumed from epoch {temp_epoch}; next epoch {start_epoch}; best_metric={best_metric:.4f}")

if RUN_TRAINING:
    run_context = mlflow.start_run(experiment_id=EXPERIMENT_ID, run_name=RUN_NAME) if MLFLOW_ENABLED else None
    if run_context is not None:
        active_run = run_context.__enter__()
        print("run_id:", active_run.info.run_id)
    try:
        if MLFLOW_ENABLED:
            mlflow.set_tags({
                "task": "mura_binary_abnormality_study_level",
                "model_family": "dinov2_end_to_end_simple_transmil",
                "backbone": BACKBONE_NAME,
                "input": "raw_study_images",
                "env": "training",
                "split_policy": "train_internal_val_90_10_by_study_official_valid_as_test",
            })
            mlflow.set_tags({f"unfreeze_epoch_{epoch}": int(blocks) for epoch, blocks in UNFREEZE_SCHEDULE.items()})
            mlflow.set_tag("augmentation", "hflip, rotation, affine, colorjitter")
            mlflow.set_tag("preprocessing", "square_pad_resize")
            mlflow.set_tag("scheduler_monitor", "internal_val_composite")
            mlflow.set_tag("checkpoint_monitor", "best_internal_val_kappa")
            mlflow.log_params({k: str(v) if isinstance(v, (dict, list, Path)) else v for k, v in training_config().items()})
            mlflow.log_artifact(str(config_path), artifact_path="setup")
            for split_csv in ARTIFACT_DIR.glob("split_*.csv"):
                mlflow.log_artifact(str(split_csv), artifact_path="setup")

        for epoch in range(start_epoch, EPOCHS + 1):
            epoch_start = time.time()
            n_blocks = configure_trainable(model, epoch)
            if optimizer is None or current_blocks != n_blocks:
                optimizer = make_optimizer(model)
                scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode="max", factor=SCHEDULER_FACTOR, patience=SCHEDULER_PATIENCE, min_lr=SCHEDULER_MIN_LR
                )
                current_blocks = n_blocks
                print(f"epoch {epoch}: configured trainable DINO blocks = {n_blocks}")

            train_loss, train_labels, train_logits, _ = run_epoch(train_loader, train=True, optimizer=optimizer, scaler=scaler)
            with torch.inference_mode():
                internal_val_loss, internal_val_labels, internal_val_logits, internal_val_pred_df = run_epoch(internal_val_loader, train=False)

            train_m = binary_metrics(train_labels, train_logits, threshold=THRESHOLD)
            internal_val_m = binary_metrics(internal_val_labels, internal_val_logits, threshold=THRESHOLD)
            best_t, best_t_m = best_threshold(internal_val_labels, internal_val_logits, metric="kappa")
            scheduler_metric = float(internal_val_m["composite"])
            if np.isfinite(scheduler_metric):
                scheduler.step(scheduler_metric)

            elapsed_min = (time.time() - epoch_start) / 60
            checkpoint_metric = float(best_t_m["kappa"])
            row = {
                "epoch": epoch,
                "elapsed_min": elapsed_min,
                "n_unfrozen_blocks": n_blocks,
                "trainable_params": trainable_param_count(model),
                "lr_groups": lr_summary(optimizer),
                "train_loss": train_loss,
                "internal_val_loss": internal_val_loss,
                "best_internal_val_threshold": best_t,
                "best_internal_val_kappa": best_t_m["kappa"],
            }
            row.update({f"train_{key}": value for key, value in train_m.items()})
            row.update({f"internal_val_{key}": value for key, value in internal_val_m.items()})
            row.update({f"best_internal_val_{key}": value for key, value in best_t_m.items()})
            history.append(row)
            pd.DataFrame(history).to_csv(history_path, index=False)

            if MLFLOW_ENABLED:
                mlflow.log_metrics({"train/loss": float(train_loss), **{f"train/{k}": v for k, v in safe_float_metrics(train_m).items()}}, step=epoch)
                mlflow.log_metrics({"internal_val/loss": float(internal_val_loss), **{f"internal_val/{k}": v for k, v in safe_float_metrics(internal_val_m).items()}}, step=epoch)
                mlflow.log_metrics({f"internal_val_best_threshold/{k}": v for k, v in safe_float_metrics(best_t_m).items()}, step=epoch)
                mlflow.log_metrics({
                    "internal_val/best_threshold": float(best_t),
                    "trainable_params": float(trainable_param_count(model)),
                    "n_unfrozen_blocks": float(n_blocks),
                    "epoch_elapsed_min": float(elapsed_min),
                }, step=epoch)

            if epoch == 1 or (np.isfinite(checkpoint_metric) and checkpoint_metric > best_metric):
                best_metric = checkpoint_metric
                best_threshold_from_internal_val = best_t
                save_checkpoint(best_checkpoint_path, optimizer, scheduler, scaler, epoch, best_metric, history, n_blocks, best_t)
                print(f"epoch {epoch}: new best internal_val kappa={best_metric:.4f} threshold={best_t:.2f}")

            save_checkpoint(last_checkpoint_path, optimizer, scheduler, scaler, epoch, best_metric, history, n_blocks, best_threshold_from_internal_val)
            print(
                f"epoch={epoch:02d}/{EPOCHS} train_loss={train_loss:.4f} internal_val_loss={internal_val_loss:.4f} "
                f"internal_val_kappa@0.5={internal_val_m['kappa']:.4f} best_kappa={best_t_m['kappa']:.4f} "
                f"pred_pos_rate@0.5={internal_val_m['pred_positive_rate']:.3f} "
                f"composite={internal_val_m['composite']:.4f} {lr_summary(optimizer)} {elapsed_min:.1f}m"
            )

        best_checkpoint = load_checkpoint(best_checkpoint_path)
        best_threshold_from_internal_val = float(best_checkpoint.get("best_internal_val_threshold", THRESHOLD))
        with torch.inference_mode():
            test_loss, test_labels, test_logits, test_pred_df = run_epoch(test_loader, train=False)

        test_at_05 = binary_metrics(test_labels, test_logits, threshold=THRESHOLD)
        test_at_best_internal = binary_metrics(test_labels, test_logits, threshold=best_threshold_from_internal_val)
        final_eval_df = pd.DataFrame([
            {"split": "test_official_valid", "threshold_source": "fixed_0.5", "threshold": THRESHOLD, "loss": test_loss, **test_at_05},
            {"split": "test_official_valid", "threshold_source": "best_internal_val_kappa", "threshold": best_threshold_from_internal_val, "loss": test_loss, **test_at_best_internal},
        ])
        final_eval_df.to_csv(final_eval_path, index=False)
        test_pred_df["pred_fixed_0.5"] = (test_pred_df["prob"] >= THRESHOLD).astype(int)
        test_pred_df["pred_best_internal_val"] = (test_pred_df["prob"] >= best_threshold_from_internal_val).astype(int)
        test_pred_df.to_csv(test_predictions_path, index=False)
        display(final_eval_df)

        if MLFLOW_ENABLED:
            final_fixed = {f"final/test_official_valid_fixed_0.5/{k}": v for k, v in safe_float_metrics(test_at_05).items()}
            final_best = {f"final/test_official_valid_best_internal_val/{k}": v for k, v in safe_float_metrics(test_at_best_internal).items()}
            mlflow.log_metrics({"final/test_official_valid/loss": float(test_loss), **final_fixed, **final_best})
            mlflow.log_artifact(str(history_path), artifact_path="training")
            mlflow.log_artifact(str(best_checkpoint_path), artifact_path="checkpoints")
            mlflow.log_artifact(str(last_checkpoint_path), artifact_path="checkpoints")
            mlflow.log_artifact(str(final_eval_path), artifact_path="final_artifacts")
            mlflow.log_artifact(str(test_predictions_path), artifact_path="final_artifacts")

            if REGISTER_MODEL:
                model_info = mlflow.pytorch.log_model(
                    model,
                    artifact_path="model",
                    registered_model_name=REGISTERED_MODEL_NAME,
                    pip_requirements=["torch", "torchvision", "mlflow>=2.15.1,<3", "numpy", "pandas", "pillow", "scikit-learn"],
                )
                client = MlflowClient()
                version = str(getattr(model_info, "registered_model_version", "") or "")
                if version:
                    client.set_model_version_tag(REGISTERED_MODEL_NAME, version, "env", "PRD")
                    client.set_model_version_tag(REGISTERED_MODEL_NAME, version, "model_family", "dinov2_end_to_end_simple_transmil")
                    client.set_model_version_tag(REGISTERED_MODEL_NAME, version, "input_type", "raw_study_images_[N,3,448,448]")
                    client.set_model_version_tag(REGISTERED_MODEL_NAME, version, "scheduler_monitor", "internal_val_composite")
                    client.set_model_version_tag(REGISTERED_MODEL_NAME, version, "checkpoint_monitor", "best_internal_val_kappa")
                    client.set_registered_model_alias(REGISTERED_MODEL_NAME, MODEL_VERSION_ALIAS, version)
                    print("registered model:", REGISTERED_MODEL_NAME, "version:", version)

        print("best internal val kappa:", best_metric)
        print("best internal val threshold:", best_threshold_from_internal_val)
        print("final official-valid-as-test eval:", final_eval_path)
    finally:
        if run_context is not None:
            run_context.__exit__(None, None, None)
else:
    print("RUN_TRAINING=False; skipping training")


/tmp/ipykernel_650/4206426455.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and DEVICE.type == "cuda")


resumed from epoch 12; next epoch 13; best_metric=0.6540
run_id: f1f519c32733402b960d03b47e595fa1


train:   0%|          | 0/12105 [00:00<?, ?it/s]

eval:   0%|          | 0/1352 [00:00<?, ?it/s]

epoch=13/30 train_loss=0.5327 internal_val_loss=0.5392 internal_val_kappa@0.5=0.6196 best_kappa=0.6352 pred_pos_rate@0.5=0.380 composite=0.8571 backbone=8.00e-06, mil_head=3.00e-04 7.9m


...

epoch=30/30 train_loss=0.3064 internal_val_loss=1.0532 internal_val_kappa@0.5=0.6576 best_kappa=0.6595 pred_pos_rate@0.5=0.345 composite=0.8512 backbone=1.00e-07, mil_head=2.34e-06 7.9m


eval:   0%|          | 0/1199 [00:00<?, ?it/s]

,split,threshold_source,threshold,loss,accuracy,precision,recall,f1,kappa,specificity,roc_auc,pr_auc,label_positive_rate,pred_positive_rate,tn,fp,fn,tp,composite
0,test_official_valid,fixed_0.5,0.5,0.580875,0.850709,0.849903,0.810409,0.829686,0.696932,0.883510,0.901923,0.893074,0.448707,0.427857,584.0,77.0,102.0,436.0,0.894843
1,test_official_valid,best_internal_val_kappa,0.9,0.580875,0.852377,0.883227,0.773234,0.824579,0.698116,0.916793,0.901923,0.893074,0.448707,0.392827,606.0,55.0,122.0,416.0,0.894843


2026/06/13 09:07:59 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'mura_dinov2_attention_trans-mil-unfreeze-backbone'.
2026/06/13 09:11:18 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: mura_dinov2_attention_trans-mil-unfreeze-backbone, version 1
Created version '1' of model 'mura_dinov2_attention_trans-mil-unfreeze-backbone'.


registered model: mura_dinov2_attention_trans-mil-unfreeze-backbone version: 1
best internal val kappa: 0.6837811842931163
best internal val threshold: 0.8999999999999999
final official-valid-as-test eval: /content/drive/MyDrive/Project2025/attention_mil_dinov2-transmil-unfreeze/results/end_to_end_transmil_dinov2_vitl14_448_final_eval.csv
🏃 View run dinov2_attention_trans_mil_unfreeze_backbone at: http://:8094/#/experiments/4/runs/f1f519c32733402b960d03b47e595fa1
🧪 View experiment at: http://:8094/#/experiments/4
